# PCAR PCTAR related studies

## Imports

In [1]:
# Imports for pythonifying code

from typing import Callable, List, Dict, Any

# Path related imports

import os
import sys

In [2]:
# Append utilities path 

current_path = os.getcwd()
cwd = os.path.dirname(current_path)

sys.path.append(os.path.join(cwd, "utilities"))

## Get input data

In [3]:
# Import pickle operations 

from pickle_file_operations import read_pickle, write_pickle

# Import dataclasses

from optimization_dataclasses import Prosumer_Cluster_Data, Market_Data, Bivariate_Pwl_Input, Layer_Data, Optimization_Results
from dataclasses import fields

## Get neural network data

In [4]:
# Relevant imports

import tensorflow as tf
from keras import layers
import numpy as np
import pandas as pd

In [5]:
def get_layer_data(layer: layers) -> Layer_Data:
    
    input_size, output_size = np.shape(layer.get_weights()[0])
        
    coefficient_matrix, bias_matrix = layer.get_weights()
    
    bias_matrix = np.reshape(bias_matrix, (1, len(bias_matrix)))
    
    activation_type = layer.activation.__name__
    
    return Layer_Data(activation_type=activation_type,
                      input_size=input_size,
                      output_size=output_size, 
                      coefficient_matrix=coefficient_matrix, 
                      bias_matrix=bias_matrix)

## Optimization model functions

In [6]:
# Import gurobi API and relevant functions

import gurobipy as gp
from gurobipy import GRB
from gurobi_ml import add_predictor_constr
import time as Time

In [7]:
def _add_market_variables(opt_model: gp.Model, time_steps: int):
    
    variable_names = ["flexibility_MWh", "updated_max_flexibility_MWh", "flexibility_cost_DKK"]
    lower_bounds = [0.098, 0.10, 0]
    upper_bounds = [GRB.INFINITY for i in range(3)]
    decision_variables = {}
    
    for time in range(time_steps):
        
        decision_variables[time] = {variable_names[i]: opt_model.addVar(vtype=GRB.CONTINUOUS, 
                                                                        lb=lower_bounds[i], ub=upper_bounds[i],
                                                                       name=variable_names[i]+"_"+str(time)) 
                                    for i in range(3)} 
        
    opt_model.update()
    
    return decision_variables

def _add_market_constraints(opt_model: gp.Model, decision_variables: Dict[str, Any],  prosumer_data: Prosumer_Cluster_Data, 
                            activation_signal: np.ndarray, epsilon: float):
    
    cross_temporal_matrix = prosumer_data.cross_temporal_matrix
    maximum_flexibility_profile = prosumer_data.max_flexibility_profile 
    time_steps = len(maximum_flexibility_profile)
    
    for time in range(time_steps):
        variables = decision_variables[time]
        
        opt_model.addConstr(variables["flexibility_MWh"] + epsilon <= variables["updated_max_flexibility_MWh"])
        opt_model.addConstr(variables["updated_max_flexibility_MWh"] 
                            == maximum_flexibility_profile[time] 
                            - (gp.quicksum(cross_temporal_matrix[j][time] * decision_variables[j]["flexibility_MWh"] 
                                           * activation_signal[j]
                                          for j in range(time_steps)
                                          )
                              )
                           )
    opt_model.update()
    
    return None


In [321]:
def _add_nn_variables(opt_model: gp.Model, time_steps: int, nn_model: tf.keras.models.Sequential):

    nn_layer_data = {}
    nn_variables = {}

    for layer_num, layer in enumerate(nn_model.layers):

        layer_data = get_layer_data(layer)
        nn_layer_data[layer_num] = layer_data
        nn_variables[layer_num] = {}
        
        for time in range(time_steps):
            
            nn_variables[layer_num][time] = {"input_layer": opt_model.addMVar(layer_data.input_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_input"), 
                                            "affine_layer": opt_model.addMVar(layer_data.output_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_affine"),
                                            "output_layer": opt_model.addMVar(layer_data.output_size, vtype=GRB.CONTINUOUS, lb=-1000, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_output"),
                                            "penalty_cost": opt_model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, 
                                                                             name="Layer_"+str(layer_num)+"_T_"+str(time)+"_penalty_cost")
                                            }
    opt_model.update()
    return nn_layer_data, nn_variables

In [322]:
def _affine_transform(opt_model: gp.Model, layer_data: Layer_Data, nn_decision_variables: Dict[str, Any]):

    opt_model.addConstr(nn_decision_variables["affine_layer"] 
                        == nn_decision_variables["input_layer"] @ layer_data.coefficient_matrix + layer_data.bias_matrix)

    opt_model.update()
    
    return None
    

def _linear_func(opt_model: gp.Model, nn_decision_variables: Dict[str, Any]):

    opt_model.addConstr(nn_decision_variables["affine_layer"] 
                        == nn_decision_variables["output_layer"])
    
    # opt_model.addConstr(nn_decision_variables["penalty_cost"] == 0)
    
    opt_model.update()
    
    return None
    
def _pctar_relu_func(opt_model: gp.Model, nn_decision_variables: Dict[str, Any], output_size: int, penalty_value: float):
    lower_bound = -10000
    upper_bound = 10000
    
    constant_1 = upper_bound / (upper_bound - lower_bound)
    constant_2 = lower_bound * constant_1
    
    opt_model.addConstr(nn_decision_variables["output_layer"] >= nn_decision_variables["affine_layer"])

    opt_model.addConstrs(nn_decision_variables["output_layer"][i] >= 0 
                         for i in range(output_size)
                        )

    opt_model.addConstrs(nn_decision_variables["output_layer"][i] 
                         <= constant_1 * nn_decision_variables["affine_layer"][i] - constant_2
                        for i in range(output_size)
                        )

    opt_model.addConstr(nn_decision_variables["penalty_cost"] 
                            == penalty_value * gp.quicksum(nn_decision_variables["output_layer"][i] 
                                                           for i in range(output_size)
                                                          )
                           )
    opt_model.update()
    return None

def _pcar_relu_func(opt_model: gp.Model, nn_decision_variables: Dict[str, Any], output_size: int, penalty_value: float):
    opt_model.addConstr(nn_decision_variables["output_layer"] >= nn_decision_variables["affine_layer"])
    opt_model.addConstrs(nn_decision_variables["output_layer"][i] >= 0 
                         for i in range(output_size)
                        )

    opt_model.addConstr(nn_decision_variables["penalty_cost"] 
                            == penalty_value * gp.quicksum(nn_decision_variables["output_layer"][i]
                                                           for i in range(output_size)
                                                          )
                           )
    opt_model.update()
    return None

In [423]:
def relu_nn_constraints(opt_model: gp.Model, nn_model: tf.keras.models.Sequential, 
                        relu_func: Callable[[gp.Model, Dict[str, Any], int], None], 
                        market_variables: Dict[str, Any], 
                        nn_scaler: Dict[str, Any], pwl_input: Bivariate_Pwl_Input, time_steps: int):
    
    nn_layer_data, nn_decision_variables = _add_nn_variables(opt_model, time_steps, nn_model)
    
    for time in range(time_steps):

        for layer_num, layer_data in nn_layer_data.items():
            _affine_transform(opt_model, layer_data, nn_decision_variables[layer_num][time])

            if layer_data.activation_type == "relu":

                relu_func(opt_model, nn_decision_variables[layer_num][time], layer_data.output_size, penalty_value)
                
            else:

                _linear_func(opt_model, nn_decision_variables[layer_num][time])
            
            if layer_num != list(nn_layer_data.keys())[-1]:  
                
                opt_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"] 
                                    == nn_decision_variables[layer_num + 1][time]["input_layer"])
                
        layer_num = list(nn_layer_data.keys())[0]

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][0] 
                            == ((market_variables[time]["flexibility_MWh"] - nn_scaler["x"]["min"]) 
                                / (nn_scaler["x"]["max"]- nn_scaler["x"]["min"])
                               )
                           )
        
        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][1] 
                            == ((market_variables[time]["updated_max_flexibility_MWh"] - nn_scaler["y"]["min"]) 
                                / (nn_scaler["y"]["max"] - nn_scaler["y"]["min"])
                               )
                           )

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][2] 
                            == ((pwl_input[time].shaping_parameters[0] - nn_scaler["a"]["min"]) 
                                / (nn_scaler["a"]["max"] - nn_scaler["a"]["min"])
                               )
                           )

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][3] 
                            == ((pwl_input[time].shaping_parameters[1] - nn_scaler["b"]["min"]) 
                                / (nn_scaler["b"]["max"] - nn_scaler["b"]["min"])
                               )
                           )
                            
        layer_num = list(nn_layer_data.keys())[-1]
                 
        opt_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"][0] 
                            == ((market_variables[time]["flexibility_cost_DKK"] - nn_scaler["z"]["min"]) 
                                / (nn_scaler["z"]["max"]- nn_scaler["z"]["min"])
                               )
                           )   

    opt_model.update()
    return nn_decision_variables

In [424]:
def makedir(path):
    if not os.path.exists(path):
        os.mkdir(path)

results_path = os.path.join(cwd, "penalty sensitivity results")
makedir(results_path)

folder_list = ["PCTAR", "PCAR"]

for folder in folder_list:
    makedir(os.path.join(results_path, folder))

In [425]:
def get_results(pwl_data, market_data, market_variables, time_steps):
    
    variable_names = ["flexibility_MWh", "updated_max_flexibility_MWh", "flexibility_cost_DKK"]
    
    results_dict = {variable_name: np.array([market_variables[time][variable_name].X for time in range(time_steps)]) 
                    for variable_name in variable_names}

    results_dict["market_price"] = market_data.market_price 
    results_dict["shaping_parameters"] = [pwl_data[time].shaping_parameters for time in range(time_steps)]
    return pd.DataFrame(results_dict)
        

## Verfication of PCAR and PCTAR models

In [ ]:
def nn_objective_function(opt_model: gp.Model, market_variables: Dict[str, Any], nn_decision_variables: Dict[str, Any], 
                          market_data: Market_Data, time_steps: int):

    layer_num_list = list(nn_decision_variables.keys())
    opt_model.setObjective(gp.quicksum(market_data.market_price[time] * market_variables[time]["flexibility_MWh"] 
                                      - 0.00002 * market_variables[time]["flexibility_cost_DKK"] for time in range(time_steps)) 
                           - gp.quicksum(nn_decision_variables[layer_num][time]["penalty_cost"] 
                                         for layer_num in layer_num_list for time in range(time_steps)), 
                           sense=GRB.MAXIMIZE)
    opt_model.update()
    return None

In [482]:
nn_path = os.path.join(cwd, "NN models")
trial_nn = tf.keras.models.load_model(os.path.join(nn_path, "Unconstrained model", "Base_NN.keras"))
time_steps = 2
# x_vals = np.linspace(5, 10, time_steps)
x_vals = np.array([[8, 10, 4, 0.52], [4, 9, 6, 0.112]])

# y_vals = trial_nn.predict(tf.convert_to_tensor(x_vals.reshape(1, time_steps)))

y_vals = trial_nn.predict(tf.convert_to_tensor(x_vals.reshape(time_steps, 4)))
print(x_vals)
print(y_vals)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
[[ 8.    10.     4.     0.52 ]
 [ 4.     9.     6.     0.112]]
[[4.050988 ]
 [2.2691455]]


In [529]:
alpha = 0.1

In [530]:
verification_model = gp.Model()
nn_layer_data, nn_decision_variables = _add_nn_variables(verification_model, time_steps, trial_nn)

for time in range(time_steps):

    for layer_num, layer_data in nn_layer_data.items():
        penalty_value = 5**-layer_num
        _affine_transform(verification_model, layer_data, nn_decision_variables[layer_num][time])

        if layer_data.activation_type == "relu": 
            _pctar_relu_func(verification_model, nn_decision_variables[layer_num][time], layer_data.output_size, penalty_value)

        else:
            _linear_func(verification_model, nn_decision_variables[layer_num][time])

        if layer_num != list(nn_layer_data.keys())[-1]:  
            verification_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"] 
                                         == nn_decision_variables[layer_num + 1][time]["input_layer"])
        
    verification_model.addConstr(nn_decision_variables[0][time]["input_layer"] == x_vals[time])

layer_last = list(nn_layer_data.keys())[-1]
# verification_model.setObjective(gp.quicksum(nn_decision_variables[layer_num][time]["penalty_cost"] 
#                                             for layer_num in nn_layer_data.keys() 
#                                             for time in range(time_steps)
#                                            ), sense=GRB.MINIMIZE)

verification_model.setObjective(alpha * gp.quicksum(nn_decision_variables[layer_last][time]["output_layer"] for time in range(time_steps)) 
                                + gp.quicksum(nn_decision_variables[layer_num][time]["penalty_cost"] 
                                            for layer_num in nn_layer_data.keys() 
                                            for time in range(time_steps)
                                           ), sense=GRB.MINIMIZE)

verification_model.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-10700T CPU @ 2.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 418 rows, 260 columns and 1640 nonzeros
Model fingerprint: 0x62120aca
Coefficient statistics:
  Matrix range     [5e-05, 1e+00]
  Objective range  [1e-01, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [2e-03, 5e+03]
Presolve removed 276 rows and 158 columns
Presolve time: 0.01s
Presolved: 142 rows, 102 columns, 1402 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -1.3377331e+02   2.538090e+02   0.000000e+00      0s
       8    1.7975195e+01   0.000000e+00   0.000000e+00      0s

Solved in 8 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.797519496e+01


In [531]:
y_pred = []
layer_num = list(nn_layer_data.keys())[-1]
for index, decision_variables in nn_decision_variables[layer_num].items():
    output_val = decision_variables["output_layer"].X
    y_pred.append(output_val[0])

for y_1, y_2 in zip(y_vals, y_pred):
    print(y_1 - y_2)
print(y_pred)

[1004.05096]
[1002.26917]
[-1000.0, -1000.0]


In [532]:
verification_model = gp.Model()
nn_layer_data, nn_decision_variables = _add_nn_variables(verification_model, time_steps, trial_nn)

for time in range(time_steps):

    for layer_num, layer_data in nn_layer_data.items():
        penalty_value = 5**-layer_num
        _affine_transform(verification_model, layer_data, nn_decision_variables[layer_num][time])

        if layer_data.activation_type == "relu": 
            _pcar_relu_func(verification_model, nn_decision_variables[layer_num][time], layer_data.output_size, penalty_value)

        else:
            _linear_func(verification_model, nn_decision_variables[layer_num][time])

        if layer_num != list(nn_layer_data.keys())[-1]:  
            verification_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"] 
                                         == nn_decision_variables[layer_num + 1][time]["input_layer"])
        
    verification_model.addConstr(nn_decision_variables[0][time]["input_layer"] == x_vals[time])

layer_last = list(nn_layer_data.keys())[-1]
# verification_model.setObjective(gp.quicksum(nn_decision_variables[layer_num][time]["penalty_cost"] 
#                                             for layer_num in nn_layer_data.keys() 
#                                             for time in range(time_steps)
#                                            ), sense=GRB.MINIMIZE)

verification_model.setObjective(alpha * gp.quicksum(nn_decision_variables[layer_last][time]["output_layer"] for time in range(time_steps)) 
                                + gp.quicksum(nn_decision_variables[layer_num][time]["penalty_cost"] 
                                            for layer_num in nn_layer_data.keys() 
                                            for time in range(time_steps)
                                           ), sense=GRB.MINIMIZE)

verification_model.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-10700T CPU @ 2.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 338 rows, 260 columns and 1480 nonzeros
Model fingerprint: 0xeed8c012
Coefficient statistics:
  Matrix range     [5e-05, 1e+00]
  Objective range  [1e-01, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [2e-03, 1e+01]
Presolve removed 338 rows and 260 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.7975195e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.797519496e+01


In [533]:
y_pred = []
layer_num = list(nn_layer_data.keys())[-1]
for index, decision_variables in nn_decision_variables[layer_num].items():
    output_val = decision_variables["output_layer"].X
    y_pred.append(output_val[0])

for y_1, y_2 in zip(y_vals, y_pred):
    print(y_1 - y_2)
print(y_pred)

[1004.05096]
[1002.26917]
[-1000.0, -1000.0]


## Sensitivity analysis

In [388]:
# Get the inputs for the optimization problem

input_path = os.path.join(cwd, "optimization input")
market_data_dict = read_pickle(os.path.join(input_path, "price_scenarios.pickle"))
pwl_data = read_pickle(os.path.join(input_path, "pwl_data_input.pickle"))
prosumer_cluster_data = read_pickle(os.path.join(input_path, "prosumer_cluster_data.pickle"))
nn_scaler = read_pickle(os.path.join(input_path, "nn_scaler.pickle"))
time_steps = 24
season = "Winter"
epsilon = 0.02

# Get NN models

unn_model = tf.keras.models.load_model(os.path.join(nn_path, "Unconstrained model", "Base_NN.keras"))
cnn_model = tf.keras.models.load_model(os.path.join(nn_path, "Constrained model", "Base_NN.keras"))

## PCTAR method

In [389]:
%%time

results_dict = {}
for scenario, market_data_list in market_data_dict.items():
    results_dict[scenario] = {}
    for case, market_data in list(enumerate(market_data_list)):
        opt_model = gp.Model()
        market_variables = _add_market_variables(opt_model, time_steps)
        _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
        nn_variables = relu_nn_constraints(opt_model, unn_model, _pctar_relu_func, market_variables, nn_scaler, pwl_data, time_steps)
        nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
    
        # Set parameters before solving
        opt_model.setParam('TimeLimit', 3600)   # 60 seconds time limit
        opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
        
    # Start measuring cpu time
        start_cpu_time = Time.process_time()
        
    # Optimize the model
        opt_model.optimize()

    # End measuring cpu time
        end_cpu_time = Time.process_time()
    
    # Save the results
        optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                    runtime=round(opt_model.Runtime, 3), 
                                                    cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                    mip_gap=opt_model.MIPGap)

        results_dict[scenario][str(case)] = optimization_results
        
        write_pickle(results_dict, os.path.join(results_path, folder_list[3], "Results.pickle"))


Set parameter TimeLimit to value 3600
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-10700T CPU @ 2.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 5088 rows, 3192 columns and 20123 nonzeros
Model fingerprint: 0xab1cda01
Coefficient statistics:
  Matrix range     [5e-05, 1e+00]
  Objective range  [2e-05, 3e+02]
  Bounds range     [1e-01, 1e+03]
  RHS range        [2e-03, 5e+03]
Presolve removed 2921 rows and 1922 columns
Presolve time: 0.03s
Presolved: 2167 rows, 1270 columns, 18401 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
     737    1.4644817e+04   0.000000e+00   0.000000e+00      0s

Solved in 737 iterations and 0.06 seconds (0.05 work units)
Optimal objective  1.464481707e+04


AttributeError: Unable to retrieve attribute 'MIPGap'

In [390]:
for time in range(time_steps):
    print(market_variables[time]["flexibility_MWh"].X)

0.098
0.098
0.098
0.098
0.098
0.098
0.098
5.798084899183059
2.452450543097197
1.7765003204541032
1.209968054768183
0.9968826597443039
0.8652231159449885
1.172899669840076
1.4740472909031987
2.3275110290572347
2.863885127691216
3.6955560488091437
4.004438382130067
6.13200379700216
6.547094069822593
6.99638785726451
3.0621760317504116
4.428149011732147


In [391]:
for time in range(time_steps):
    print(market_variables[time]["flexibility_cost_DKK"].X)

0.0
0.0
0.0
0.0
0.0
0.0
0.0
77.15275298403937
15.987163499578742
10.061265431187028
11.804429096825476
6.284492814245214
10.522164002048816
7.329767507076884
25.35339093943025
29.955572813581608
6.416094125419577
82.06112982721847
15.026852678086923
35.326487110788285
34.64911347200367
37.2757058842339
36.521436995817226
31.54715843579608


## PCAR method

In [392]:
%%time

results_dict = {}
for scenario, market_data_list in market_data_dict.items():
    results_dict[scenario] = {}
    for case, market_data in list(enumerate(market_data_list)):
        opt_model = gp.Model()
        market_variables = _add_market_variables(opt_model, time_steps)
        _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
        nn_variables = relu_nn_constraints(opt_model, unn_model, _pcar_relu_func, market_variables, nn_scaler, pwl_data, time_steps)
        nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
    
    # Set parameters before solving
        opt_model.setParam('TimeLimit', 3600)   # 60 seconds time limit
        opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
        
    # Start measuring cpu time
        start_cpu_time = Time.process_time()
        
    # Optimize the model
        opt_model.optimize()

    # End measuring cpu time
        end_cpu_time = Time.process_time()
    
    # Save the results
        optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                    runtime=round(opt_model.Runtime, 3), 
                                                    cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                    mip_gap=opt_model.MIPGap)

        results_dict[scenario][str(case)] = optimization_results
        
        write_pickle(results_dict, os.path.join(results_path, folder_list[4], "Results.pickle"))


Set parameter TimeLimit to value 3600
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-10700T CPU @ 2.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 4128 rows, 3192 columns and 18203 nonzeros
Model fingerprint: 0x254812f4
Coefficient statistics:
  Matrix range     [5e-05, 1e+00]
  Objective range  [2e-05, 3e+02]
  Bounds range     [1e-01, 1e+03]
  RHS range        [2e-03, 1e+01]
Presolve removed 3212 rows and 2231 columns
Presolve time: 0.01s
Presolved: 916 rows, 961 columns, 9774 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
     617    1.4644817e+04   0.000000e+00   0.000000e+00      0s

Solved in 617 iterations and 0.02 seconds (0.02 work units)
Optimal objective  1.464481707e+04


AttributeError: Unable to retrieve attribute 'MIPGap'

In [393]:
for time in range(time_steps):
    print(market_variables[time]["flexibility_cost_DKK"].X)

0.0
0.0
0.0
0.0
0.0
0.0
0.0
77.15275298403937
15.987163499578742
10.061265431187028
11.804429096825476
6.284492814245214
10.522164002048816
7.329767507076884
25.35339093943025
29.955572813581608
6.416094125419577
82.06112982721847
15.026852678086923
35.326487110788285
34.64911347200367
37.2757058842339
36.521436995817226
31.54715843579608
